# CerviGrade: PLIP linear probe on CRIC and Mendeley

PLIP is a vision-language model already trained on pathology images and their captions. In this project it is evaluated by **linear probing**: PLIP itself is never changed. Every image is passed through PLIP once to get 512 numbers that describe it (the features), and then one small classifier is trained on those numbers. This is the cheapest way to see how much a pathology-trained model already knows about cervical cells, and it is the method the proposal names for PLIP.

This notebook is separate from the Stage 2 notebook on purpose. PLIP needs Internet switched on to download its weights, and switching settings can restart a Kaggle session and wipe the trained CNNs. Here, PLIP runs on its own, saves its results as files, and the Stage 2 notebook picks those files up as an input.

**Before running**
- Add these inputs: `cervigrade-cric-dataset` and `Mendeley LBC Cervical Cancer`. SIPaKMeD and the Training notebook are not needed.
- Notebook settings: turn **GPU on** and **Internet on** before running anything.
- Run the cells top to bottom. Then use **Save Version, Save & Run All** so the output files become available to other notebooks.

**What it saves** (in `/kaggle/working`)
- `plip_cric_results.json`: PLIP's scores on the CRIC test split, in the same format as the other models' results.
- `plip_cric_probs.npz`: PLIP's predicted probabilities for every CRIC test cell, with the file names so they line up with the other models.
- `plip_mendeley_results.json`: PLIP's whole-photo scores on Mendeley.
- `plip_cric_probe.joblib` and `plip_*_features.npz`: the trained probe and the extracted features, so nothing has to be recomputed.

## Find the datasets

Looks through the attached inputs for the CRIC crops and the Mendeley photos, the same way the Stage 2 notebook does.

In [ ]:
import os

def find_containing(root, predicate, max_matches=5):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if predicate(dirpath, dirnames, filenames):
            matches.append(dirpath)
            if len(matches) >= max_matches:
                break
    return matches

INPUT_ROOT = "/kaggle/input"

cric_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: os.path.basename(d) == "CRIC_cropped")
print("CRIC_cropped found at:", cric_matches)
CRIC_CROPPED_DIR = cric_matches[0]

mendeley_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: "Negative for Intraepithelial malignancy" in dn)
print("Mendeley root candidates:", mendeley_matches)
MENDELEY_ROOT = mendeley_matches[0]

## Build the CRIC and Mendeley manifests and the same train / validation / test split

This is the same code and the same random seed (42) as the Stage 2 notebook, so PLIP is scored on exactly the same 1,723 CRIC test cells as EfficientNetB0, ConvNeXt and RegNet. Whole source images stay together on one side of the split.

In [ ]:
import re
import random
import numpy as np
import pandas as pd

RANDOM_SEED = 42

def build_cric_manifest():
    rows = []
    for cls in sorted(os.listdir(CRIC_CROPPED_DIR)):
        cls_dir = os.path.join(CRIC_CROPPED_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            slide_id = fname.split("_")[0]
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "CRIC", "group": f"CRIC_slide_{slide_id}"})
    return pd.DataFrame(rows)

def build_mendeley_manifest():
    rows = []
    for cls in sorted(os.listdir(MENDELEY_ROOT)):
        cls_dir = os.path.join(MENDELEY_ROOT, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            photo_id = re.split(r"\s*\(", fname)[0].strip()
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "Mendeley", "group": f"Mendeley_{photo_id}"})
    return pd.DataFrame(rows)

def allocate_group_counts(n, train_frac=0.7, val_frac=0.15, test_frac=0.15):
    if n == 1:
        return 1, 0, 0
    if n == 2:
        return 1, 0, 1
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac)) if n - n_test >= 2 else 0
    n_train = n - n_test - n_val
    if n_train < 1:
        n_train = 1
        if n_val > 0:
            n_val -= 1
        else:
            n_test -= 1
    return n_train, n_val, n_test

def split_by_group_per_class(manifest_df, seed=RANDOM_SEED):
    rng = random.Random(seed)
    group_split = {}
    for label, sub in manifest_df.groupby("label"):
        groups = sorted(sub["group"].unique())
        rng.shuffle(groups)
        n_train, n_val, n_test = allocate_group_counts(len(groups))
        for g in groups[:n_train]:
            group_split[g] = "train"
        for g in groups[n_train:n_train + n_val]:
            group_split[g] = "val"
        for g in groups[n_train + n_val:n_train + n_val + n_test]:
            group_split[g] = "test"
    return manifest_df["group"].map(group_split)

def split_by_group_global(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(manifest_df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train = round(n * train_frac)
    n_val = round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]:
        group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]:
        group_split[g] = "val"
    for g in groups[n_train + n_val:]:
        group_split[g] = "test"
    return manifest_df["group"].map(group_split)

cric_manifest = build_cric_manifest()
mendeley_manifest = build_mendeley_manifest()
cric_manifest["split"] = split_by_group_global(cric_manifest)
mendeley_manifest["split"] = split_by_group_per_class(mendeley_manifest)

cric_classes = sorted(cric_manifest["label"].unique())
cric_class_to_idx = {c: i for i, c in enumerate(cric_classes)}
SHORT_NAMES = [c.replace("Negative for intraepithelial lesion", "Negative") for c in cric_classes]

cric_train_df = cric_manifest[cric_manifest["split"] == "train"]
cric_val_df = cric_manifest[cric_manifest["split"] == "val"]
cric_test_df = cric_manifest[cric_manifest["split"] == "test"]
print(f"CRIC  train {len(cric_train_df)}  val {len(cric_val_df)}  test {len(cric_test_df)}   (the test count should be 1,723)")
print("Mendeley split table:")
print(mendeley_manifest.groupby(["label", "split"]).size().unstack(fill_value=0))

## Load PLIP

PLIP downloads from Hugging Face (`vinid/plip`), so Internet must be on. Nothing inside PLIP is trained here, so every weight is frozen.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

PLIP_ID = "vinid/plip"
plip_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", plip_device)

try:
    plip_model = CLIPModel.from_pretrained(PLIP_ID).to(plip_device).eval()
    plip_processor = CLIPProcessor.from_pretrained(PLIP_ID)
except Exception as e:
    raise RuntimeError("Could not download PLIP. Check that Internet is switched on in the notebook settings, "
                       "then run this cell again.") from e

for p in plip_model.parameters():
    p.requires_grad = False
n_params = sum(p.numel() for p in plip_model.parameters())
print(f"PLIP loaded. {n_params / 1e6:.0f} million parameters, all frozen.")

## Describe every image with 512 numbers

Each image goes through PLIP's own preprocessing (resize, centre crop, normalise) and then through PLIP, which returns 512 numbers. A progress bar shows each split. The features are saved as they finish.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class PlipImages(Dataset):
    def __init__(self, filepaths):
        self.filepaths = list(filepaths)
    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, i):
        image = Image.open(self.filepaths[i]).convert("RGB")
        return plip_processor(images=image, return_tensors="pt")["pixel_values"][0]

def plip_features(filepaths, desc, batch_size=64):
    loader = DataLoader(PlipImages(filepaths), batch_size=batch_size, shuffle=False, num_workers=2)
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(plip_device.type == "cuda")):
        for batch in tqdm(loader, desc=desc, unit="batch"):
            feats = plip_model.get_image_features(pixel_values=batch.to(plip_device))
            if not torch.is_tensor(feats):      # newer transformers versions wrap the result
                feats = feats.pooler_output
            feats = torch.nn.functional.normalize(feats.float(), dim=-1)
            out.append(feats.cpu().numpy())
    return np.concatenate(out)

plip_X = {name: plip_features(df["filepath"], f"PLIP features, CRIC ({name})")
          for name, df in [("train", cric_train_df), ("val", cric_val_df), ("test", cric_test_df)]}
np.savez("/kaggle/working/plip_cric_features.npz", **plip_X)
print({k: v.shape for k, v in plip_X.items()})

## Train the linear probe on CRIC and score it

One small classifier (logistic regression) is trained on PLIP's 512 numbers. Its strength setting C is chosen by the lowest validation loss, then it is scored on the CRIC test cells in the same format as the other three models. The class order is the same alphabetical order used everywhere else in the project.

In [ ]:
import json
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, classification_report, accuracy_score

y_train = cric_train_df["label"].map(cric_class_to_idx).values
y_val = cric_val_df["label"].map(cric_class_to_idx).values
y_true = cric_test_df["label"].map(cric_class_to_idx).values

scaler = StandardScaler().fit(plip_X["train"])
X_train, X_val, X_test = [scaler.transform(plip_X[k]) for k in ("train", "val", "test")]

# C controls how strongly the probe is held back from memorising the training set (smaller = stronger).
C_GRID = [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]
best_loss, best_C, plip_probe = None, None, None
for C in tqdm(C_GRID, desc="Trying settings for C", unit="fit"):
    probe = LogisticRegression(C=C, max_iter=3000).fit(X_train, y_train)
    val_loss = log_loss(y_val, probe.predict_proba(X_val), labels=list(range(len(cric_classes))))
    tqdm.write(f"C={C}: validation loss {val_loss:.4f}")
    if best_loss is None or val_loss < best_loss:
        best_loss, best_C, plip_probe = val_loss, C, probe
print(f"Chosen C = {best_C}")

y_prob_plip = plip_probe.predict_proba(X_test)
y_pred = y_prob_plip.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=SHORT_NAMES, digits=3, zero_division=0))
report = classification_report(y_true, y_pred, target_names=SHORT_NAMES, output_dict=True, zero_division=0)

plip_results = {"PLIP": {
    "accuracy": round(accuracy_score(y_true, y_pred), 3),
    "weighted_f1": round(report["weighted avg"]["f1-score"], 3),
    "macro_f1": round(report["macro avg"]["f1-score"], 3),
    **{f"recall_{c}": round(report[c]["recall"], 3) for c in SHORT_NAMES},
}}
with open("/kaggle/working/plip_cric_results.json", "w") as f:
    json.dump(plip_results, f, indent=2)

# File names go in the probabilities file so the Stage 2 notebook can line the rows up with its own test cells.
test_keys = (cric_test_df["label"] + "/" + cric_test_df["filepath"].map(os.path.basename)).values
np.savez("/kaggle/working/plip_cric_probs.npz", test_keys=test_keys, y_true=y_true, y_prob=y_prob_plip)
joblib.dump({"scaler": scaler, "probe": plip_probe, "classes": cric_classes}, "/kaggle/working/plip_cric_probe.joblib")
print(plip_results)

## PLIP on Mendeley (whole photos)

The same idea on the Mendeley photos. Each image is a whole microscope field with one label for the photo. PLIP shrinks and centre-crops each photo to 224 pixels, so small cells lose detail, and the rare classes have only a few source photos each. Read these numbers as rough.

In [ ]:
mendeley_classes = sorted(mendeley_manifest["label"].unique())
mendeley_idx = {c: i for i, c in enumerate(mendeley_classes)}

def short_label(name):
    n = name.lower()
    if "negative" in n: return "Negative"
    if "high" in n: return "HSIL"
    if "low" in n: return "LSIL"
    if "carcinoma" in n or "squamous" in n: return "SCC"
    return name[:20]
mendeley_short = [short_label(c) for c in mendeley_classes]

parts = {s: mendeley_manifest[mendeley_manifest["split"] == s] for s in ("train", "val", "test")}
print({s: len(d) for s, d in parts.items()}, "images in train / val / test")
mx = {s: plip_features(d["filepath"], f"PLIP features, Mendeley ({s})", batch_size=32) for s, d in parts.items() if len(d)}
my = {s: d["label"].map(mendeley_idx).values for s, d in parts.items() if len(d)}
np.savez("/kaggle/working/plip_mendeley_features.npz", **mx)

m_scaler = StandardScaler().fit(mx["train"])
Xm = {s: m_scaler.transform(x) for s, x in mx.items()}

best_loss, best_C, m_probe = None, None, None
for C in tqdm(C_GRID, desc="Trying settings for C", unit="fit"):
    probe = LogisticRegression(C=C, max_iter=3000).fit(Xm["train"], my["train"])
    loss = log_loss(my["val"], probe.predict_proba(Xm["val"]), labels=list(probe.classes_)) if "val" in Xm else 0.0
    tqdm.write(f"C={C}: validation loss {loss:.4f}")
    if best_loss is None or loss < best_loss:
        best_loss, best_C, m_probe = loss, C, probe
print(f"Chosen C = {best_C}")

pred = m_probe.predict(Xm["test"])
labels_all = list(range(len(mendeley_classes)))
print(classification_report(my["test"], pred, labels=labels_all, target_names=mendeley_short, digits=3, zero_division=0))
m_report = classification_report(my["test"], pred, labels=labels_all, target_names=mendeley_short, output_dict=True, zero_division=0)
mendeley_results = {"PLIP": {
    "accuracy": round(accuracy_score(my["test"], pred), 3),
    "weighted_f1": round(m_report["weighted avg"]["f1-score"], 3),
    "macro_f1": round(m_report["macro avg"]["f1-score"], 3),
    **{f"recall_{c}": round(m_report[c]["recall"], 3) for c in mendeley_short},
    "test_images": int(len(pred)),
}}
with open("/kaggle/working/plip_mendeley_results.json", "w") as f:
    json.dump(mendeley_results, f, indent=2)
print(mendeley_results)
print("Treat this as a rough read: a few images move every number.")

## Check what was saved

Lists the output files. After **Save Version, Save & Run All**, these become available to the Stage 2 notebook through **Add Input, Notebook Output**.

In [ ]:
import time
for f in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", f)
    if os.path.isfile(p):
        print(f"{f:36s} {os.path.getsize(p)/1e6:8.2f} MB  {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))}")

## How to add PLIP to the Stage 2 notebook

1. Finish this notebook with **Save Version, Save & Run All**.
2. Open the Stage 2 notebook, click **Add Input, Notebook Output**, and pick this notebook. Check its folder name in the Input panel.
3. Paste the cell below near the end of Stage 2. It adds PLIP's row to `results` and lines PLIP's probabilities up with the Stage 2 test cells. After it runs, the comparison table and graphs include PLIP.

```python
import json
PLIP_DIR = "/kaggle/input/notebooks/wagnermushayija/cervigrade-plip"   # change to the folder name shown in the Input panel

results["PLIP"] = json.load(open(f"{PLIP_DIR}/plip_cric_results.json"))["PLIP"]

d = np.load(f"{PLIP_DIR}/plip_cric_probs.npz", allow_pickle=True)
row_of = {k: i for i, k in enumerate(d["test_keys"])}
keys = (cric_test_df["label"] + "/" + cric_test_df["filepath"].map(os.path.basename)).values
rows = [row_of[k] for k in keys]
y_prob_plip = d["y_prob"][rows]
assert (d["y_true"][rows] == np.asarray(y_true)).all(), "PLIP rows do not match this notebook's test cells"
print("PLIP added.", results["PLIP"])
```